In [ ]:
import os
import numpy as np
import pandas as pd

CSV_PATH = '../prepr_data/dns_smartphones_cleaned.csv'
src = pd.read_csv(CSV_PATH)
src['phone_id'] = src['id'] + 1
src = src.drop(columns=['id'])
print(src.shape)

(1376, 84)


## 1. Описание схемы
- **Справочники** : таблица → колонка исходного файла, значения которой выносятся в справочник. Справочник material общий для материала задней панели и граней.
- **Характеристики** : для каждой таблицы список (колонка в БД, колонка в источнике, тип). Типы: i — целое, n — дробное, b — флаг (boolean), fk:<справочник> — ссылка на справочник.
- Пропуски остаются пропусками и попадают в БД как `NULL`

In [17]:
DICTS = {
    'brand': ['brand'], 'color': ['color'], 'os': ['os'], 'form_factor': ['form_factor'],
    'panel_type': ['panel_type'], 'notch_type': ['notch_type'], 'screen_glass': ['screen_glass'],
    'material': ['back_material', 'frame_material'],
    'ram_type': ['ram_type'], 'battery_type': ['battery_type'], 'flash_type': ['flash_type'],
    'gpu_vendor': ['gpu_vendor'], 'cpu_vendor': ['cpu_vendor'], 'cpu_family': ['cpu_family'],
}
VIDEO_RANK = {'hd': 1, 'full hd': 2, 'quad hd': 3, 'ultra hd 4k': 4, 'ultra hd 8k': 5}

SPECS = {
    'phone_display': [
        ('diag', 'diag', 'n'), ('screen_w', 'screen_w', 'i'), ('screen_h', 'screen_h', 'i'),
        ('brightness_nits', 'brightness_nits', 'i'), ('refresh_hz', 'refresh_hz', 'i'),
        ('color_depth_bits', 'color_depth_bits', 'i'),
        ('panel_type_id', 'panel_type', 'fk:panel_type'), ('notch_type_id', 'notch_type', 'fk:notch_type'),
        ('screen_glass_id', 'screen_glass', 'fk:screen_glass'),
        ('curved_screen', 'curved_screen', 'b'), ('bezelless', 'bezelless', 'b')],
    'phone_body': [
        ('width_mm', 'width_mm', 'n'), ('height_mm', 'height_mm', 'n'), ('thickness_mm', 'thickness_mm', 'n'),
        ('weight_g', 'weight_g', 'n'),
        ('back_material_id', 'back_material', 'fk:material'), ('frame_material_id', 'frame_material', 'fk:material'),
        ('ip_dust', 'ip_dust', 'i'), ('ip_water', 'ip_water', 'i'), ('rugged', 'rugged', 'b')],
    'phone_performance': [
        ('cpu_id', 'cpu_model', 'fk:cpu'), ('cpu_cores', 'cpu_cores', 'i'), ('process_nm', 'process_nm', 'i'),
        ('cpu_ghz', 'cpu_ghz', 'n'), ('gpu_vendor_id', 'gpu_vendor', 'fk:gpu_vendor'),
        ('ram_type_id', 'ram_type', 'fk:ram_type'), ('os_version', 'os_version', 'i'),
        ('is_gaming', 'is_gaming', 'b')],
    'phone_camera_rear': [
        ('rear_cams', 'rear_cams', 'i'), ('has_telephoto', 'has_telephoto', 'b'), ('has_periscope', 'has_periscope', 'b'),
        ('has_ultrawide', 'has_ultrawide', 'b'), ('has_macro', 'has_macro', 'b'), ('digital_zoom', 'digital_zoom', 'i'),
        ('video_res_id', 'video_max_res', 'fk:video_res'), ('flash_type_id', 'flash_type', 'fk:flash_type'),
        ('ois', 'ois', 'b'), ('main_autofocus', 'main_autofocus', 'b'), ('slow_motion', 'slow_motion', 'b')],
    'phone_camera_front': [
        ('front_autofocus', 'front_autofocus', 'b'), ('front_flash', 'front_flash', 'b'),
        ('dual_front_cam', 'dual_front_cam', 'b')],
    'phone_power': [
        ('battery_mah', 'battery_mah', 'i'), ('battery_type_id', 'battery_type', 'fk:battery_type'),
        ('charger_in_box', 'charger_in_box', 'b'), ('charger_w_in_box', 'charger_w_in_box', 'n'),
        ('fast_charge', 'fast_charge', 'b'), ('wired_charge_w', 'wired_charge_w', 'n'),
        ('wireless_charge', 'wireless_charge', 'b'), ('wireless_charge_w', 'wireless_charge_w', 'n'),
        ('reverse_charge', 'reverse_charge', 'b'), ('reverse_wireless', 'reverse_wireless', 'b')],
    'phone_connectivity': [
        ('has_5g', 'has_5g', 'b'), ('lte_advanced', 'lte_advanced', 'b'), ('volte', 'volte', 'b'),
        ('sim_physical', 'sim_physical', 'i'), ('esim_count', 'esim_count', 'i'), ('nfc', 'nfc', 'b'),
        ('bluetooth_ver', 'bluetooth_ver', 'n'), ('wifi_gen', 'wifi_gen', 'i'),
        ('n_nav_systems', 'n_nav_systems', 'i'), ('ir_port', 'ir_port', 'b')],
    'phone_features': [
        ('otg', 'otg', 'b'), ('fm_radio', 'fm_radio', 'b'), ('memory_card_slot', 'memory_card_slot', 'b'),
        ('stereo_speakers', 'stereo_speakers', 'b'), ('gms', 'gms', 'b'), ('headphone_jack', 'headphone_jack', 'b'),
        ('face_unlock', 'face_unlock', 'b'), ('fingerprint', 'fingerprint', 'b')],
}

# колонки источника, которые идут не в SPECS-таблицы, а в phone_model / phone / cpu (нужно только для проверки полноты)
OTHER_SRC = {'phone_id', 'brand', 'model', 'color', 'memory', 'ram_gb', 'price', 'useful_life',
             'year_release', 'form_factor', 'os', 'cpu_model', 'cpu_vendor', 'cpu_family'}
used = OTHER_SRC | {s for cols in SPECS.values() for _, s, _ in cols}
print('колонок источника не попало в схему:', sorted(set(src.columns) - used))

колонок источника не попало в схему: []


## 2. Сборка таблиц

In [18]:
def cast(s, kind):
    if kind == 'i':
        return s.astype('Int64')
    if kind == 'b':
        return s.astype('boolean')
    return s.astype('float64')


def make_dict(values, pk):
    vals = sorted(pd.unique(pd.concat([pd.Series(v) for v in values]).dropna()))
    return pd.DataFrame({pk: np.arange(1, len(vals) + 1), 'name': vals})


def fk(series, dic, pk):
    """название -> id справочника; пропуск остаётся пропуском"""
    return series.map(dict(zip(dic['name'], dic[pk]))).astype('Int64')


T = {}   # имя таблицы -> DataFrame

# --- справочники
for name, cols in DICTS.items():
    T[name] = make_dict([src[c] for c in cols], f'{name}_id')
T['video_res'] = pd.DataFrame({'video_res_id': np.arange(1, 6), 'name': list(VIDEO_RANK),
                               'quality_rank': list(VIDEO_RANK.values())})
unknown = set(src['video_max_res'].dropna()) - set(VIDEO_RANK)
assert not unknown, f'неизвестные форматы видео: {unknown}'

# --- процессоры (вендор и линейка определяются моделью чипа)
cpu_src = src.dropna(subset=['cpu_model'])
assert (cpu_src.groupby('cpu_model')[['cpu_vendor', 'cpu_family']].nunique() <= 1).all().all()
cpu = cpu_src.groupby('cpu_model', as_index=False)[['cpu_vendor', 'cpu_family']].first()
T['cpu'] = pd.DataFrame({
    'cpu_id': np.arange(1, len(cpu) + 1), 'name': cpu['cpu_model'],
    'cpu_vendor_id': fk(cpu['cpu_vendor'], T['cpu_vendor'], 'cpu_vendor_id'),
    'cpu_family_id': fk(cpu['cpu_family'], T['cpu_family'], 'cpu_family_id')})

# --- модель телефона (бренд + название)
pm = src[['brand', 'model']].drop_duplicates().sort_values(['brand', 'model']).reset_index(drop=True)
pm['model_id'] = np.arange(1, len(pm) + 1)
T['phone_model'] = pd.DataFrame({
    'model_id': pm['model_id'],
    'brand_id': fk(pm['brand'], T['brand'], 'brand_id'),
    'name': pm['model']})

# --- товар (карточка)
model_id = src.merge(pm[['brand', 'model', 'model_id']], on=['brand', 'model'], how='left')['model_id']
T['phone'] = pd.DataFrame({
    'phone_id': src['phone_id'], 'model_id': model_id.values,
    'color_id': fk(src['color'], T['color'], 'color_id'),
    'memory_gb': cast(src['memory'], 'i'), 'ram_gb': cast(src['ram_gb'], 'i'),
    'price': cast(src['price'], 'i'), 'useful_life': cast(src['useful_life'], 'i'),
    'year_release': cast(src['year_release'], 'i'),
    'form_factor_id': fk(src['form_factor'], T['form_factor'], 'form_factor_id'),
    'os_id': fk(src['os'], T['os'], 'os_id')})

# --- характеристики 1:1
DICT_PK = {**{n: f'{n}_id' for n in DICTS}, 'video_res': 'video_res_id'}
for table, cols in SPECS.items():
    out = {'phone_id': src['phone_id']}
    for target, source, kind in cols:
        if kind.startswith('fk:'):
            d = kind[3:]
            if d == 'cpu':      # ссылка на процессор по названию чипа
                out[target] = fk(src[source], T['cpu'], 'cpu_id')
            else:
                out[target] = fk(src[source], T[d], DICT_PK[d])
        else:
            out[target] = cast(src[source], kind)
    T[table] = pd.DataFrame(out)

LOAD_ORDER = (list(DICTS) + ['video_res', 'cpu', 'phone_model', 'phone'] + list(SPECS))
print(pd.Series({t: len(T[t]) for t in LOAD_ORDER}, name='строк').to_string())

brand                   26
color                   18
os                       2
form_factor              3
panel_type               2
notch_type               4
screen_glass             9
material                 9
ram_type                 4
battery_type             3
flash_type               4
gpu_vendor               6
cpu_vendor               7
cpu_family              20
video_res                5
cpu                    128
phone_model            296
phone                 1376
phone_display         1376
phone_body            1376
phone_performance     1376
phone_camera_rear     1376
phone_camera_front    1376
phone_power           1376
phone_connectivity    1376
phone_features        1376


## 3. Проверки перед загрузкой

In [ ]:
# 1) каждая связь ссылается на существующую запись справочника
FK_CHECKS = [('phone_model', 'brand_id', 'brand'), ('phone', 'model_id', 'phone_model'), ('phone', 'color_id', 'color'),
             ('phone', 'form_factor_id', 'form_factor'), ('phone', 'os_id', 'os'),
             ('cpu', 'cpu_vendor_id', 'cpu_vendor'), ('cpu', 'cpu_family_id', 'cpu_family')]
for table, cols in SPECS.items():
    for target, source, kind in cols:
        if kind.startswith('fk:'):
            FK_CHECKS.append((table, target, kind[3:]))
for table, col, ref in FK_CHECKS:
    ref_pk = T[ref].columns[0]
    bad = ~T[table][col].dropna().isin(T[ref][ref_pk])
    assert not bad.any(), f'{table}.{col}: есть значения, которых нет в {ref}'

# 2) по одной строке характеристик на каждый телефон
for table in SPECS:
    assert T[table]['phone_id'].is_unique and len(T[table]) == len(T['phone']), table

пропуски сохранены в 78 колонках; всего пустых ячеек: 5551


## 4. Подключение через Spark

In [20]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, ShortType, IntegerType, LongType, DoubleType,
                               BooleanType, StringType)
import os
from getpass import getpass

os.environ.pop('SPARK_HOME', None)

PG_HOST = os.getenv('PG_HOST', 'localhost')
PG_PORT = os.getenv('PG_PORT', '5433')
PG_DB = os.getenv('PG_DB', 'dns_smartphones')
PG_USER = os.getenv('PG_USER', 'postgres')
PG_PASSWORD = os.getenv('PG_PASSWORD') or getpass('Пароль PostgreSQL: ')
PG_SCHEMA = 'smartphones'

JDBC_URL = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}'
JDBC_PROPS = {'user': PG_USER, 'password': PG_PASSWORD, 'driver': 'org.postgresql.Driver'}

spark = (SparkSession.builder
         .appName('dns_smartphones_load')
         .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.4')
         .getOrCreate())
print(JDBC_URL)

jdbc:postgresql://localhost:5433/dns_smartphones


## 5. Создание схемы

In [ ]:
RECREATE_SCHEMA = True
SCHEMA_SQL = 'schema.sql'

if RECREATE_SCHEMA:
    with open(SCHEMA_SQL, encoding='utf-8') as f:
        ddl = f.read()

    jvm = spark._sc._gateway.jvm
    props = jvm.java.util.Properties()
    props.setProperty('user', PG_USER)
    props.setProperty('password', PG_PASSWORD)
    conn = jvm.org.postgresql.Driver().connect(JDBC_URL, props) 
    try:
        stmt = conn.createStatement()
        stmt.execute(ddl)  
        stmt.close()
    finally:
        conn.close()
    print('схема', PG_SCHEMA, 'создана заново')

схема smartphones создана заново


## 6. Загрузка данных
Таблицы записываются через JDBC в режиме append строго в порядке зависимостей: справочники -> модель -> телефоны -> характеристики. После загрузки число строк в каждой таблице сверяется с pandas.

In [ ]:
def spark_type(dtype):
    name = str(dtype)
    if name == 'boolean' or name == 'bool':
        return BooleanType()
    if name in ('Int8', 'Int16', 'int8', 'int16'):
        return ShortType()
    if name in ('Int32', 'int32'):
        return IntegerType()
    if name in ('Int64', 'int64'):
        return LongType()
    if pd.api.types.is_float_dtype(dtype):
        return DoubleType()
    return StringType()


def to_py(v):
    if v is None or v is pd.NA or (isinstance(v, float) and np.isnan(v)):
        return None
    return v.item() if hasattr(v, 'item') else v


def to_spark(df):
    schema = StructType([StructField(c, spark_type(t), True) for c, t in df.dtypes.items()])
    rows = [tuple(to_py(v) for v in r) for r in df.itertuples(index=False, name=None)]
    return spark.createDataFrame(rows, schema)

In [25]:
for table in LOAD_ORDER:
    to_spark(T[table]).write.jdbc(url=JDBC_URL, table=f'{PG_SCHEMA}.{table}', mode='append', properties=JDBC_PROPS)
    print('загружено:', table, len(T[table]))

загружено: brand 26
загружено: color 18
загружено: os 2
загружено: form_factor 3
загружено: panel_type 2
загружено: notch_type 4
загружено: screen_glass 9
загружено: material 9
загружено: ram_type 4
загружено: battery_type 3
загружено: flash_type 4
загружено: gpu_vendor 6
загружено: cpu_vendor 7
загружено: cpu_family 20
загружено: video_res 5
загружено: cpu 128
загружено: phone_model 296
загружено: phone 1376
загружено: phone_display 1376
загружено: phone_body 1376
загружено: phone_performance 1376
загружено: phone_camera_rear 1376
загружено: phone_camera_front 1376
загружено: phone_power 1376
загружено: phone_connectivity 1376
загружено: phone_features 1376


In [26]:
# проверка
for table in LOAD_ORDER:
    n_db = spark.read.jdbc(url=JDBC_URL, table=f'{PG_SCHEMA}.{table}', properties=JDBC_PROPS).count()
    assert n_db == len(T[table]), f'{table}: в БД {n_db}, загружено {len(T[table])}'
print('все таблицы загружены полностью')

все таблицы загружены полностью
